Los datos han sido obtenidos de la siguiente url:  
https://www.kaggle.com/datasets/computingvictor/transactions-fraud-datasets  
Este 'dataset' consta de 3 .csv y de 2 .json
Nosotros juntaremos los datos de estas 5 tablas en una unica tabla
Eliminaremos aquellas variables que no aporten mucho al modelo o su cardinalidad es equivalente
Por un lado seleccionaremos solo aquellos datos que tengan etiqueta de si hay o no fraude para evaluar el modelo
Por otro lado dividiremos nuestros datos por años, 2010-2017 y 2018-2019 para simular una ingesta de datos en tiempo real con Kafka

In [1]:
import pyspark
from pyspark.sql import SparkSession

spark = (SparkSession.builder 
    .appName("ETL")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "4g")
    .getOrCreate()
)

print("Spark version:", spark.version)

Spark version: 3.5.0


In [2]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, MapType, IntegerType, DoubleType, TimestampType
import json
import pandas as pd
pd.set_option('display.max_columns', None)
path = "/home/jovyan/work/data/"

Importamos los datos de las transacciones y hacemos las transformaciones pertinentes

In [3]:
transactions_data_schema = StructType([
    StructField("id", StringType(), True),
    StructField("date", TimestampType(), True),
    
    StructField("client_id", StringType(), True),
    StructField("card_id", StringType(), True),   
    StructField("amount", StringType(), True),   
    
    StructField("use_chip", StringType(), True),
    StructField("merchant_id", IntegerType(), True),
    StructField("merchant_city", StringType(), True),
    StructField("merchant_state", StringType(), True),
    StructField("zip", StringType(), True),
    StructField("mcc", StringType(), True),
    StructField("errors", StringType(), True)
])

transactions_data_df = (spark.read
    .option("header", "true") 
    .schema(transactions_data_schema)
    .csv(f"{path}/raw/transactions_data.csv")
    .withColumns({
        "amount": F.regexp_replace(F.col("amount"), r"\$", "").cast("double"),
        "zip": F.lpad(F.regexp_replace(F.col("zip").cast("string"), r"\.0$", ""), 5, "0")
    })
)

transactions_data_df.printSchema()
transactions_data_df.limit(5).toPandas()

root
 |-- id: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- client_id: string (nullable = true)
 |-- card_id: string (nullable = true)
 |-- amount: double (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- mcc: string (nullable = true)
 |-- errors: string (nullable = true)



,id,date,client_id,card_id,amount,use_chip,merchant_id,merchant_city,merchant_state,zip,mcc,errors
0,7475327,2010-01-01 00:01:00,1556,2972,-77.00,Swipe Transaction,59935,Beulah,ND,58523,5499,None
1,7475328,2010-01-01 00:02:00,561,4575,14.57,Swipe Transaction,67570,Bettendorf,IA,52722,5311,None
2,7475329,2010-01-01 00:02:00,1129,102,80.00,Swipe Transaction,27092,Vista,CA,92084,4829,None
3,7475331,2010-01-01 00:05:00,430,2860,200.00,Swipe Transaction,27092,Crown Point,IN,46307,4829,None
4,7475332,2010-01-01 00:06:00,848,3915,46.41,Swipe Transaction,13051,Harwood,MD,20776,5813,None


In [8]:
transactions_data_df = transactions_data_df.distinct()
transactions_data_df.write.mode("overwrite").save(f"{path}silver/transactions_data_df.parquet")

Importamos los datos de los usuarios y hacemos las transformaciones pertinentes

In [5]:
users_data_schema = StructType([       
    StructField("client_id", StringType(), True),                 
    StructField("current_age", IntegerType(), True),
    StructField("retirement_age", IntegerType(), True),
    StructField("birth_year", IntegerType(), True),
    StructField("birth_month", IntegerType(), True),

    StructField("gender", StringType(), True),
    StructField("address", StringType(), True),

    StructField("client_latitude", DoubleType(), True),
    StructField("client_longitude", DoubleType(), True),

    StructField("per_capita_income", StringType(), True),  
    StructField("yearly_income", StringType(), True),      
    StructField("total_debt", StringType(), True), 

    StructField("credit_score", IntegerType(), True),
    StructField("num_credit_cards", IntegerType(), True)       
])

users_data_df = (spark.read
    .option("header", "true") 
    .schema(users_data_schema)
    .csv(f"{path}/raw/users_data.csv")
    
    .withColumns({
        "per_capita_income": F.regexp_replace(F.col("per_capita_income"), r"\$", "").cast("double"),
        "yearly_income": F.regexp_replace(F.col("yearly_income"), r"\$", "").cast("double"),
        "total_debt": F.regexp_replace(F.col("total_debt"), r"\$", "").cast("double")
    })
)

users_data_df.printSchema()
users_data_df.limit(5).toPandas()

root
 |-- client_id: string (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- client_latitude: double (nullable = true)
 |-- client_longitude: double (nullable = true)
 |-- per_capita_income: double (nullable = true)
 |-- yearly_income: double (nullable = true)
 |-- total_debt: double (nullable = true)
 |-- credit_score: integer (nullable = true)
 |-- num_credit_cards: integer (nullable = true)



,client_id,current_age,retirement_age,birth_year,birth_month,gender,address,client_latitude,client_longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards
0,825,53,66,1966,11,Female,462 Rose Lane,34.15,-117.76,29278.0,59696.0,127613.0,787,5
1,1746,53,68,1966,12,Female,3606 Federal Boulevard,40.76,-73.74,37891.0,77254.0,191349.0,701,5
2,1718,81,67,1938,11,Female,766 Third Drive,34.02,-117.89,22681.0,33483.0,196.0,698,5
3,708,63,63,1957,1,Female,3 Madison Street,40.71,-73.99,163145.0,249925.0,202328.0,722,4
4,1164,43,70,1976,9,Male,9620 Valley Stream Drive,37.76,-122.44,53797.0,109687.0,183855.0,675,1


In [6]:
users_data_df = users_data_df.distinct()
users_data_df.write.mode("overwrite").save(f"{path}silver/users_data_df.parquet")

Importamos los datos de las tarjetas y hacemos las transformaciones pertinentes

In [5]:
cards_data_schema = StructType([
    
    StructField("card_id", StringType(), True),                 
    StructField("client_id", StringType(), True),   
    StructField("card_brand", StringType(), True),
    StructField("card_type", StringType(), True),
    StructField("card_number", StringType(), True), 
    
    StructField("expires", StringType(), True),
    StructField("cvv", StringType(), True),
    StructField("has_chip", StringType(), True),
    StructField("num_cards_issued", IntegerType(), True),

    StructField("credit_limit", StringType(), True), 
    
    StructField("acct_open_date", StringType(), True),
    StructField("year_pin_last_changed", IntegerType(), True),
    StructField("card_on_dark_web", StringType(), True)        
])

cards_data_df = (spark.read
    .option("header", "true")
    .schema(cards_data_schema)
    .csv(f"{path}raw/cards_data.csv")

    .withColumns({
        "has_chip": F.upper(F.col("has_chip")) == "YES",
        "card_on_dark_web": F.upper(F.col("card_on_dark_web")) == "YES",
        
        "credit_limit": F.regexp_replace(F.col("credit_limit"), r"\$", "").cast("double"),
        "expires": F.to_date(F.concat(F.lit("01/"), F.col("expires")), "dd/MM/yyyy"),
        "acct_open_date": F.to_date(F.concat(F.lit("01/"), F.col("acct_open_date")), "dd/MM/yyyy")
    })
)

cards_data_df.printSchema()
cards_data_df.limit(5).toPandas()

root
 |-- card_id: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- card_brand: string (nullable = true)
 |-- card_type: string (nullable = true)
 |-- card_number: string (nullable = true)
 |-- expires: date (nullable = true)
 |-- cvv: string (nullable = true)
 |-- has_chip: boolean (nullable = true)
 |-- num_cards_issued: integer (nullable = true)
 |-- credit_limit: double (nullable = true)
 |-- acct_open_date: date (nullable = true)
 |-- year_pin_last_changed: integer (nullable = true)
 |-- card_on_dark_web: boolean (nullable = true)



,card_id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
0,4524,825,Visa,Debit,4344676511950444,2022-12-01,623,True,2,24295.0,2002-09-01,2008,False
1,2731,825,Visa,Debit,4956965974959986,2020-12-01,393,True,2,21968.0,2014-04-01,2014,False
2,3701,825,Visa,Debit,4582313478255491,2024-02-01,719,True,2,46414.0,2003-07-01,2004,False
3,42,825,Visa,Credit,4879494103069057,2024-08-01,693,False,1,12400.0,2003-01-01,2012,False
4,4659,825,Mastercard,Debit (Prepaid),5722874738736011,2009-03-01,75,True,1,28.0,2008-09-01,2009,False


In [12]:
cards_data_df = cards_data_df.distinct()
cards_data_df.write.mode("overwrite").save(f"{path}silver/cards_data_df.parquet")

Importamos los datos de los códigos de comercio y hacemos las transformaciones pertinentes

In [14]:
with open(f"{path}/raw/mcc_codes.json", "r", encoding="utf-8") as mcc_f:
    mcc_fragment = mcc_f.read(400)
    print(mcc_fragment)

{
    "5812": "Eating Places and Restaurants",
    "5541": "Service Stations",
    "7996": "Amusement Parks, Carnivals, Circuses",
    "5411": "Grocery Stores, Supermarkets",
    "4784": "Tolls and Bridge Fees",
    "4900": "Utilities - Electric, Gas, Water, Sanitary",
    "5942": "Book Stores",
    "5814": "Fast Food Restaurants",
    "4829": "Money Transfer",
    "5311": "Department Stores",
   


In [10]:
mcc_codes_df = (spark.read
    .option("multiline", "true")
    .json(f"{path}raw/mcc_codes.json")
    .select(F.from_json(F.to_json(F.struct("*")), "map<string,string>").alias("dictionary_mcc"))
    .select(F.explode("dictionary_mcc").alias("mcc", "mcc_description"))
)

mcc_codes_df.printSchema()
mcc_codes_df.limit(5).toPandas()

root
 |-- mcc: string (nullable = false)
 |-- mcc_description: string (nullable = true)



,mcc,mcc_description
0,1711,"Heating, Plumbing, Air Conditioning Contractors"
1,3000,Steelworks
2,3001,Steel Products Manufacturing
3,3005,Miscellaneous Metal Fabrication
4,3006,Miscellaneous Fabricated Metal Products


In [14]:
mcc_codes_df = mcc_codes_df.distinct()
mcc_codes_df.write.mode("overwrite").save(f"{path}silver/mcc_codes_df.parquet")

Importamos los datos de las etiquetas de fraude y hacemos las transformaciones pertinentes

In [12]:
with open(f"{path}/raw/train_fraud_labels.json", "r", encoding="utf-8") as lbl_f:
    lbl_fragment = lbl_f.read(400)
    print(lbl_fragment)

{"target": {"10649266": "No", "23410063": "No", "9316588": "No", "12478022": "No", "9558530": "No", "12532830": "No", "19526714": "No", "9906964": "No", "13224888": "No", "13749094": "No", "12303776": "No", "19480376": "No", "11716050": "No", "20025400": "No", "7661688": "No", "16662807": "No", "21419778": "No", "18011186": "No", "23289598": "No", "11644547": "No", "23235120": "No", "19748218": "N


In [3]:
##Ejecutamos el siguiente codigo en la terminal para transformar el json a ndjson
#jq -c '.target | to_entries[] | {id: .key, target: .value}' train_fraud_labels.json > train_fraud_labels.ndjson

/bin/bash: line 1: jq: command not found


In [15]:
train_fraud_labels_schema = StructType([
    StructField("id", StringType(), True),
    StructField("target", StringType(), True)
])


train_fraud_labels_df = (spark.read
    .schema(train_fraud_labels_schema)
    .json(f"{path}raw/train_fraud_labels.ndjson")
    .withColumn("target", F.upper(F.col("target")) == "YES")
)

train_fraud_labels_df.printSchema()
train_fraud_labels_df.limit(5).toPandas()

root
 |-- id: string (nullable = true)
 |-- target: boolean (nullable = true)



,id,target
0,10649266,False
1,23410063,False
2,9316588,False
3,12478022,False
4,9558530,False


In [16]:
train_fraud_labels_df = train_fraud_labels_df.distinct()
train_fraud_labels_df.write.mode("overwrite").save(f"{path}silver/train_fraud_labels_df.parquet")

En esta sección procedemos a hacer el merge de nuestros datos

In [3]:
!ls $path/silver

cards_data_df.parquet  train_fraud_labels_df.parquet  users_data_df.parquet
mcc_codes_df.parquet   transactions_data_df.parquet


In [5]:
import pgeocode
import geonamescache

transactions_data_df = spark.read.parquet(f"{path}silver/transactions_data_df.parquet")
mcc_codes_df = spark.read.parquet(f"{path}silver/mcc_codes_df.parquet")
users_data_df = spark.read.parquet(f"{path}silver/users_data_df.parquet")
cards_data_df = spark.read.parquet(f"{path}silver/cards_data_df.parquet")
train_fraud_labels_df = spark.read.parquet(f"{path}silver/train_fraud_labels_df.parquet")

En esta sección añadiremos coordenadas al código zip de los comercios

In [14]:
zip_df = transactions_data_df.select("zip").distinct().collect()
zip_list = [row["zip"] for row in zip_df]

nomi = pgeocode.Nominatim('us') 
geo_data = nomi.query_postal_code(zip_list)

geo_zip_dict = {
    row['postal_code']: (row['latitude'], row['longitude'])
    for _, row in geo_data.dropna(subset=['latitude', 'longitude']).iterrows()
}

geo_zip_list = [(zip_code, coords[0], coords[1]) for zip_code, coords in geo_zip_dict.items()]
geo_zip_df = spark.createDataFrame(geo_zip_list, 
                                   schema=["zip", "merchant_latitude", "merchant_longitude"])

geo_zip_df.printSchema()
geo_zip_df.limit(5).toPandas()

root
 |-- zip: string (nullable = true)
 |-- merchant_latitude: double (nullable = true)
 |-- merchant_longitude: double (nullable = true)



,zip,merchant_latitude,merchant_longitude
0,77303,30.3814,-95.3749
1,75007,33.0033,-96.8820
2,93924,36.4787,-121.7244
3,32773,28.7644,-81.2820
4,25555,38.2300,-82.5753


In [16]:
city_state_df = (
    transactions_data_df
    .filter(
        (F.col("zip").isNull()) &
        (F.col("merchant_city") != "ONLINE")
    )
    .select("merchant_city", "merchant_state").distinct().collect()
)

gc = geonamescache.GeonamesCache()
geo_city_dict = {}

corrections = {
    "Sao Paolo": "São Paulo", "Bishek": "Bishkek", "Johannesberg": "Johannesburg",
    "Guatamala City": "Guatemala City", "Tapei": "Taipei", "Tblisi": "Tbilisi"
}

for row in city_state_df:
    city = row["merchant_city"]
    state = row["merchant_state"] 
    if not city:
        continue      
    city_corrected = corrections.get(city, city)
    match = gc.get_cities_by_name(city_corrected)
    
    if match:
        all_cities = [list(c.values())[0] for c in match]
        match_city = None
        for c in all_cities:
            if c.get('admin1code') == state or c.get('countrycode') == state:
                match_city = c
                break
        if not match_city:
            match_city = max(all_cities, key=lambda x: x.get('population', 0))
        geo_city_dict[(city, state)] = (float(match_city['latitude']), float(match_city['longitude']))



geo_city_list = [(item[0], item[1], coords[0], coords[1]) for item, coords in geo_city_dict.items()]
geo_city_df = spark.createDataFrame(
    geo_list, 
    schema=["merchant_city", "merchant_state", "city_latitude", "city_longitude"]
)

geo_city_df.printSchema()
geo_city_df.limit(5).toPandas()


root
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- city_latitude: double (nullable = true)
 |-- city_longitude: double (nullable = true)



,merchant_city,merchant_state,city_latitude,city_longitude
0,Sao Paolo,Brazil,-23.54750,-46.63611
1,Oranjestad,Aruba,12.52398,-70.02703
2,Zagreb,Croatia,45.81444,15.97798
3,Manama,Bahrain,26.22787,50.58565
4,Male,Maldives,4.17521,73.50916


In [18]:
complete_fraud_data_df = (transactions_data_df
    .join(geo_zip_df, on="zip", how="left")
    .join(geo_city_df, on=["merchant_city", "merchant_state"], how="left")
    .withColumns({
        "merchant_latitude": F.coalesce(F.col("merchant_latitude"), F.col("city_latitude")),
        "merchant_longitude": F.coalesce(F.col("merchant_longitude"), F.col("city_longitude"))
    })
    .drop("city_latitude", "city_longitude")
    .join(mcc_codes_df, on="mcc", how="left")
    .join(users_data_df, on="client_id", how="left")
    .join(cards_data_df, on=["client_id", "card_id"], how="left")
    .join(train_fraud_labels_df, on="id", how="left")
)

Filas = complete_fraud_data_df.count()
Columnas = len(complete_fraud_data_df.columns)

print(f"Filas: {Filas}, Columnas: {Columnas}")
complete_fraud_data_df.printSchema()
complete_fraud_data_df.limit(5).toPandas()

Filas: 13305915, Columnas: 40
root
 |-- id: string (nullable = true)
 |-- client_id: string (nullable = true)
 |-- card_id: string (nullable = true)
 |-- mcc: string (nullable = true)
 |-- merchant_city: string (nullable = true)
 |-- merchant_state: string (nullable = true)
 |-- zip: string (nullable = true)
 |-- date: timestamp (nullable = true)
 |-- amount: double (nullable = true)
 |-- use_chip: string (nullable = true)
 |-- merchant_id: integer (nullable = true)
 |-- errors: string (nullable = true)
 |-- merchant_latitude: double (nullable = true)
 |-- merchant_longitude: double (nullable = true)
 |-- mcc_description: string (nullable = true)
 |-- current_age: integer (nullable = true)
 |-- retirement_age: integer (nullable = true)
 |-- birth_year: integer (nullable = true)
 |-- birth_month: integer (nullable = true)
 |-- gender: string (nullable = true)
 |-- address: string (nullable = true)
 |-- client_latitude: double (nullable = true)
 |-- client_longitude: double (nullable = t

,id,client_id,card_id,mcc,merchant_city,merchant_state,zip,date,amount,use_chip,merchant_id,errors,merchant_latitude,merchant_longitude,mcc_description,current_age,retirement_age,birth_year,birth_month,gender,address,client_latitude,client_longitude,per_capita_income,yearly_income,total_debt,credit_score,num_credit_cards,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web,target
0,7522049,1209,5888,3389,Spring Valley,NY,10977,2010-01-12 21:27:00,126.62,Swipe Transaction,16790,Bad PIN,41.1158,-74.0474,Non-Precious Metal Services,45,67,1974,6,Female,95 12th Drive,40.94,-73.86,31299.0,63815.0,4832.0,699,6,Mastercard,Debit,5471110680954467,2020-01-01,175,True,2,42074.0,2007-12-01,2010,False,None
1,7504572,1133,2586,4829,Perry,NY,14530,2010-01-08 12:43:00,120.00,Swipe Transaction,27092,Insufficient Balance,42.7229,-78.0059,Money Transfer,30,66,1989,10,Female,781 Federal Street,43.11,-77.80,22855.0,46599.0,12700.0,684,4,Visa,Debit,4874200877830198,2023-04-01,234,True,2,21442.0,2008-04-01,2015,False,False
2,7494622,1895,2119,4829,New York,NY,10010,2010-01-05 20:54:00,100.00,Swipe Transaction,27092,Insufficient Balance,40.7375,-73.9813,Money Transfer,56,65,1963,3,Female,900 Littlewood Street,39.75,-74.22,20034.0,40848.0,146608.0,706,2,Mastercard,Debit,5267518025716847,2022-01-01,292,True,2,2042.0,2006-03-01,2013,False,None
3,7478118,1940,3859,5813,Corona,NY,11368,2010-01-01 15:45:00,19.55,Swipe Transaction,81922,Technical Glitch,40.7453,-73.8611,Drinking Places (Alcoholic Beverages),40,65,1979,10,Female,464 Lake Drive,40.74,-73.85,13177.0,26865.0,19529.0,826,5,Mastercard,Credit,5405403307724984,2020-02-01,44,True,2,9100.0,2008-07-01,2008,False,None
4,7489384,734,4990,5411,Buffalo,NY,14211,2010-01-04 13:56:00,61.58,Swipe Transaction,15137,Insufficient Balance,42.9082,-78.8225,"Grocery Stores, Supermarkets",43,61,1976,7,Female,57799 Oak Boulevard,42.88,-78.85,12220.0,24917.0,36101.0,693,2,Mastercard,Credit,5146631536689214,2020-07-01,498,True,2,6300.0,2003-10-01,2010,False,None


In [19]:
complete_fraud_data_df.write.mode("overwrite").save(f"{path}bronze/complete_fraud_data_df.parquet")